#1. GPU 检查

In [ ]:
!nvidia-smi # GPU 检查

#2. 安装项目依赖

这些库分别用于加载 Qwen、处理数据、GPU 运行、LoRA 和 SFT。Qwen3 要求 Transformers 至少为 4.51.0。

In [ ]:
# 固定已验证的依赖版本
%pip install \
    transformers==5.22.1 \
    datasets==5.0.1 \
    pyarrow==23.0.1 \
    accelerate==1.15.0 \
    peft==0.21.2 \
    trl==1.17.1

# 3. 依赖验证
就是检查安装好的工具能不能正常使用

1.能否导入：例如 Transformers、PEFT、TRL 有没有安装或兼容问题。

2.版本是多少：记录实际环境，方便以后复现。

3.GPU 是否可用：确认 PyTorch 能使用 Colab 的显卡。

In [ ]:
import torch
import transformers
import datasets
import pyarrow
import accelerate
import peft
import trl

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

for module in [torch, transformers, datasets, pyarrow, accelerate, peft, trl]:
    print(f"{module.__name__}: {module.__version__}")

print("GPU 可用:", torch.cuda.is_available())
print("核心依赖导入成功")

#4. 固定随机种子

作用是让实验更容易重复、比较更公平。减少后续实验因随机性产生的差异。

模型训练有一些随机操作，例如：
- 训练数据的打乱顺序。
- LoRA 新增参数的初始化。
- 训练时 Dropout 随机屏蔽部分神经元。
设置 SEED = 42，相当于给这些操作一个固定的起点。以后比较不同 LoRA 参数时，可以减少随机变化的干扰。

它不会直接提高准确率，也不会降低 Loss。

In [ ]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("随机种子:", SEED)

#5. 加载 Qwen3-0.6B
我们使用官方模型 Qwen/Qwen3-0.6B，通过 Transformers 加载

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-0.6B"

# 加载分词器：负责文本与 token 之间的转换
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 加载原始模型，使用 FP16 并放到 GPU 上
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map={"": 0},
)

model.eval()

print("模型加载成功")
print("模型名称:", MODEL_ID)
print("运行设备:", model.device)
print("参数精度:", model.dtype)
print(f"参数量: {model.num_parameters() / 1e9:.3f} B")

# 6. 物流信息提取推理测试

In [ ]:
# 虚构示例，仅用于验证推理流程
prompt = """
请从以下物流记录中提取信息，只输出 JSON，不要添加解释。
字段：车牌号、出发地、目的地、运费收入、燃油费、路桥费、异常原因。
金额使用数字；未提供的信息填 null，不要编造。

物流记录：
川A12345从重庆运货到成都，本趟运费收入3500元，
燃油费900元，路桥费450元。途中因暴雨延误2小时。
"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# 只显示模型新生成的内容
new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(answer)

**原始模型推理验证成功。**这条示例中，7 个字段均提取正确，金额为数字，没有编造信息。

有一点需要记录：模型在 JSON 外加了 Markdown 代码围栏（```json）。内容正确，但如果要求程序直接解析整个输出，它还不属于严格的纯 JSON。这可以作为后续微调和评测的观察点。

## Phase 0：环境搭建与原始模型推理验证

- 项目：FleetMind-LLM
- 本地电脑：Intel MacBook Pro，16GB 内存
- 运行平台：Google Colab
- GPU：NVIDIA Tesla T4，15360 MiB 显存
- Python：3.13.16
- PyTorch：2.11.0+cu130
- PyTorch CUDA：13.0
- Transformers：5.22.1
- Datasets：5.0.1
- PyArrow：23.0.1
- Accelerate：1.15.0
- PEFT：0.21.2
- TRL：1.17.1

### 模型与推理设置
- 原始模型：Qwen/Qwen3-0.6B
- 参数量：约 0.596B
- 运行设备：cuda:0
- 参数精度：FP16
- 思考模式：关闭
- 解码方式：贪心解码（do_sample=False）
- 最大新生成 token 数：256

### 验证结果
- 核心依赖导入成功，GPU 可用。
- 模型加载成功，完成一条虚构物流记录的信息提取。
- 本条示例的 7 个字段均提取正确。
- 输出带有 Markdown 代码围栏，尚未满足严格纯 JSON 要求。
- 本次仅验证推理流程，不代表正式测试集表现。
- 物流领域数据集待确定。